# 文書要約のベースラインと ROUGE 評価

書籍 第4章「言語生成問題の解き方」（4-1 文書要約）に対応する演習ノートブックです。
CPU のみ・数秒で実行でき、以下の内容を体験します。

1. 要約の評価尺度 **ROUGE-1 / ROUGE-2 / ROUGE-L** の定義式からの実装（4-1-4）
2. 抽出型要約の代表手法（**リード法**・**TF-IDF スコアリング**・**TextRank**）の実装と比較（4-1-1, 4-1-2）
3. 記事構成（逆ピラミッド型 vs 後半結論型）による手法ごとの得意・不得意の分析

## 1. 準備とサンプル記事データの作成

日本語の文字単位（または簡易バイグラム）で評価・抽出を行うため、追加パッケージは不要で、標準ライブラリと `numpy`・`pandas`・`scikit-learn` のみで動作します。

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ニュース記事風の「冒頭に結論がある記事（逆ピラミッド型）」と、解説・ブログ風の「中盤〜後半に結論がある記事」を混ぜた 6 本のサンプル記事（各 5 文）と人手参照要約を用意します。

In [2]:
articles = [
    {
        "id": 1,
        "type": "ニュース型（冒頭結論）",
        "sentences": [
            "政府は本日、AI分野の研究開発を支援する新たな補助金制度を来年度から開始すると発表しました。",
            "対象となるのは大学やスタートアップ企業で、最大で1件あたり5000万円が支給されます。",
            "近年、生成AIの急速な普及に伴い、国内での計算資源や人材の不足が課題となっていました。",
            "関係省庁は先月から有識者会議を開き、具体的な支援枠組みについて議論を重ねてきました。",
            "申請の受付は来年4月から開始される予定で、詳細は公式ウェブサイトで公開されます。",
        ],
        "reference": "政府は大学やスタートアップ向けに最大5000万円を支給するAI研究開発の新たな補助金制度を来年度から開始します。",
    },
    {
        "id": 2,
        "type": "ニュース型（冒頭結論）",
        "sentences": [
            "東京都内の主要鉄道各社は、来月1日から終電時刻を約15分繰り上げるダイヤ改正を実施します。",
            "これは深夜の保線作業の時間を確保し、作業員の負担を軽減することが主な目的です。",
            "対象となるのは山手線や中央線などの主要路線で、始発時刻に変更はありません。",
            "利用者の減少傾向も背景にあり、コロナ禍以降、深夜帯の乗車率は約3割低下していました。",
            "各社は駅構内のポスターやアプリを通じて、利用者への周知を進めています。",
        ],
        "reference": "都内の主要鉄道各社は深夜の保線作業時間確保のため、来月1日から終電を約15分繰り上げます。",
    },
    {
        "id": 3,
        "type": "ニュース型（冒頭結論）",
        "sentences": [
            "大手電機メーカーのA社は、従来比で充電時間を半分に短縮した新型バッテリーを開発しました。",
            "新開発の電極材料を採用することで、発熱を抑えながら急速充電を実現しています。",
            "同社は2年前から国立研究所と共同で次世代電池の基礎研究を進めてきました。",
            "この技術は電気自動車やスマートフォンへの応用が期待されており、年内の量産開始を目指します。",
            "発表を受けて、本日の株式市場ではA社の株価が一時5%上昇しました。",
        ],
        "reference": "A社が充電時間を半分に短縮した新型バッテリーを開発し、年内の量産開始を目指しています。",
    },
    {
        "id": 4,
        "type": "後半結論型（課題→調査→結論）",
        "sentences": [
            "私たちの街では近年、駅前商店街の空き店舗が増加していることが大きな悩みでした。",
            "以前は休日に多くの家族連れで賑わっていましたが、郊外に大型商業施設ができて状況が一変しました。",
            "市役所は現状を把握するため、住民2000人を対象に買い物行動に関するアンケート調査を実施しました。",
            "調査の結果、若年層は夜間の営業やカフェスペースの充実を求めていることが明らかになりました。",
            "これを受けて市は、空き店舗を改装したコワーキングスペースと夜カフェの誘致を最優先施策として決定しました。",
        ],
        "reference": "住民調査の結果を踏まえ、市は商店街の空き店舗にコワーキングスペースと夜カフェを誘致する施策を決定しました。",
    },
    {
        "id": 5,
        "type": "後半結論型（実験→比較→結論）",
        "sentences": [
            "自然言語処理のモデル開発では、最初から大規模な深層学習モデルを使うべきか議論になることがあります。",
            "大規模モデルは高い精度が期待できる一方、GPUコストや推論速度の面で運用が難しくなる場合があります。",
            "そこで今回、同じ文書分類データに対してルールベース、TF-IDF、BERTの3手法を比較実験しました。",
            "実験では1000件のサンプルデータを用い、正解率と推論時間、誤分類の傾向を詳しく分析しました。",
            "その結果、まずはTF-IDFなどの軽量なベースラインでデータ品質を確認してからBERTへ進むのが最も効率的だとわかりました。",
        ],
        "reference": "比較実験の結果、文書分類ではTF-IDFなどの軽量なベースラインでデータを確認してからBERTへ進むのが最も効率的だとわかりました。",
    },
    {
        "id": 6,
        "type": "後半結論型（背景→問題→結論）",
        "sentences": [
            "リモートワークが定着したことで、オンライン会議の回数が以前より大幅に増えています。",
            "会議の録音や議事録作成の負担を軽減するため、音声認識ツールの導入を検討する企業が増えました。",
            "しかし、専門用語や社内略語が多い会議では、汎用モデルそのままでは誤変換が多発してしまいます。",
            "実際に試したところ、固有名詞の誤認識が原因で議事録の手直しに多くの時間がかかりました。",
            "実用化には、社内の固有表現をユーザー辞書に登録し、さらに要約モデルで要点を整理する仕組みが不可欠です。",
        ],
        "reference": "オンライン会議の議事録自動化を実用化するには、社内固有名詞の辞書登録と要約モデルによる要点整理が不可欠です。",
    },
]
print(f"記事数: {len(articles)} 本")

記事数: 6 本


---

## 2. 評価尺度 ROUGE の実装（4-1-4「自動評価」）

文書要約では、参照要約に含まれる重要な情報（単語・フレーズ）をシステム要約がどれだけ**漏れなく拾えているか（再現率; Recall）**を重視する **ROUGE** が標準的に使われます。

- **ROUGE-1 / ROUGE-2**: 1-gram / 2-gram の重なりに基づく評価（日本語では文字 n-gram または単語 n-gram を用います。ここではトークナイザに依存しない文字 n-gram で計算します）
- **ROUGE-L**: 語順を保った最長共通部分列（Longest Common Subsequence; **LCS**）の長さに基づく評価

ここでは ROUGE-1, ROUGE-2, ROUGE-L の F1 スコアを定義式から実装します。

In [3]:
from collections import Counter

def char_ngrams(text, n):
    t = text.replace(" ", "").replace("　", "")
    return [t[i:i + n] for i in range(len(t) - n + 1)]

def rouge_n_f1(hyp, ref, n=1):
    h_ng = Counter(char_ngrams(hyp, n))
    r_ng = Counter(char_ngrams(ref, n))
    overlap = sum(min(cnt, r_ng[g]) for g, cnt in h_ng.items())
    h_total = sum(h_ng.values())
    r_total = sum(r_ng.values())
    if overlap == 0 or h_total == 0 or r_total == 0:
        return 0.0
    prec = overlap / h_total
    rec = overlap / r_total
    return 2 * prec * rec / (prec + rec)

def lcs_length(a, b):
    '''動的計画法で文字列 a, b の最長共通部分列（LCS）の長さを求める'''
    m, n = len(a), len(b)
    dp = [0] * (n + 1)
    for i in range(m):
        prev = 0
        for j in range(n):
            temp = dp[j + 1]
            if a[i] == b[j]:
                dp[j + 1] = prev + 1
            else:
                dp[j + 1] = max(dp[j + 1], dp[j])
            prev = temp
    return dp[n]

def rouge_l_f1(hyp, ref):
    h = hyp.replace(" ", "").replace("　", "")
    r = ref.replace(" ", "").replace("　", "")
    if not h or not r:
        return 0.0
    lcs = lcs_length(h, r)
    if lcs == 0:
        return 0.0
    prec = lcs / len(h)
    rec = lcs / len(r)
    return 2 * prec * rec / (prec + rec)

# 動作確認
print("ROUGE-1:", round(rouge_n_f1("政府はAIの補助金を開始します", "政府はAI研究開発の補助金を開始します", 1), 3))
print("ROUGE-L:", round(rouge_l_f1("政府はAIの補助金を開始します", "政府はAI研究開発の補助金を開始します"), 3))

ROUGE-1: 0.882
ROUGE-L: 0.882


---

## 3. 抽出型要約の 3 手法を実装・比較する（4-1-1, 4-1-2）

記事の 5 文から **最も重要な 1 文（Lead-1 相当）** を抽出するタスクとして、次の 3 手法を比較します。

1. **リード法（Lead）**: 常に先頭の 1 文目を選ぶ単純なベースライン
2. **TF-IDF スコアリング**: 文字バイグラムの TF-IDF ベクトルにおいて、文書全体（全 5 文の重心）に最も近い文、または重要語重みの合計が高い文を選ぶ
3. **TextRank**: 各文をノード、文間の TF-IDF コサイン類似度をエッジ重みとするグラフ上で PageRank（べき乗法）を実行し、中心性の高い文を選ぶ

In [4]:
def summarize_lead(sentences, k=1):
    return "".join(sentences[:k])

def summarize_tfidf(sentences, k=1):
    vec = TfidfVectorizer(analyzer="char", ngram_range=(2, 3))
    mat = vec.fit_transform(sentences).toarray()
    # 文書全体の重心ベクトルとの類似度が高い文を選ぶ
    doc_vec = mat.mean(axis=0, keepdims=True)
    sims = cosine_similarity(mat, doc_vec).ravel()
    top_idx = sorted(np.argsort(sims)[::-1][:k])
    return "".join(sentences[i] for i in top_idx)

def summarize_textrank(sentences, k=1, damping=0.85, n_iter=30):
    vec = TfidfVectorizer(analyzer="char", ngram_range=(2, 3))
    mat = vec.fit_transform(sentences)
    sim = cosine_similarity(mat)
    np.fill_diagonal(sim, 0.0)
    row_sums = sim.sum(axis=1, keepdims=True)
    # 孤立ノード対策
    trans = np.divide(sim, row_sums, out=np.full_like(sim, 1.0 / len(sentences)), where=row_sums != 0)
    scores = np.ones(len(sentences)) / len(sentences)
    for _ in range(n_iter):
        scores = (1.0 - damping) / len(sentences) + damping * trans.T @ scores
    top_idx = sorted(np.argsort(scores)[::-1][:k])
    return "".join(sentences[i] for i in top_idx)

methods = {
    "リード法": summarize_lead,
    "TF-IDF": summarize_tfidf,
    "TextRank": summarize_textrank,
}

eval_rows = []
for art in articles:
    for m_name, fn in methods.items():
        hyp = fn(art["sentences"], k=1)
        eval_rows.append({
            "記事ID": art["id"],
            "構成タイプ": art["type"],
            "手法": m_name,
            "ROUGE-1": round(rouge_n_f1(hyp, art["reference"], 1), 3),
            "ROUGE-2": round(rouge_n_f1(hyp, art["reference"], 2), 3),
            "ROUGE-L": round(rouge_l_f1(hyp, art["reference"]), 3),
            "抽出された文": hyp,
        })

res_df = pd.DataFrame(eval_rows)
print("=== 全記事の平均 ROUGE スコア ===")
display(res_df.groupby("手法")[["ROUGE-1", "ROUGE-2", "ROUGE-L"]].mean().round(3))

print("\n=== 記事の構成タイプ別の平均 ROUGE-L ===")
display(res_df.pivot_table(index="構成タイプ", columns="手法", values="ROUGE-L", aggfunc="mean").round(3))

=== 全記事の平均 ROUGE スコア ===


,ROUGE-1,ROUGE-2,ROUGE-L
手法,,,
TF-IDF,0.449,0.255,0.340
TextRank,0.449,0.255,0.340
リード法,0.515,0.343,0.431



=== 記事の構成タイプ別の平均 ROUGE-L ===


手法,TF-IDF,TextRank,リード法
構成タイプ,,,
ニュース型（冒頭結論）,0.351,0.351,0.620
後半結論型（実験→比較→結論）,0.342,0.342,0.191
後半結論型（背景→問題→結論）,0.280,0.280,0.232
後半結論型（課題→調査→結論）,0.367,0.367,0.304


### 記事ごとの抽出結果を確認する

スコアだけでなく、実際にどの文が選ばれたかを参照要約と見比べてみましょう。

In [5]:
for art in articles:
    print(f"[記事 {art['id']}: {art['type']}]")
    print(f"  参照要約: {art['reference']}")
    sub = res_df[res_df["記事ID"] == art["id"]]
    for _, row in sub.iterrows():
        print(f"  {row['手法']:8s} (R-L={row['ROUGE-L']:.3f}): {row['抽出された文']}")
    print()

[記事 1: ニュース型（冒頭結論）]
  参照要約: 政府は大学やスタートアップ向けに最大5000万円を支給するAI研究開発の新たな補助金制度を来年度から開始します。
  リード法     (R-L=0.549): 政府は本日、AI分野の研究開発を支援する新たな補助金制度を来年度から開始すると発表しました。
  TF-IDF   (R-L=0.208): 申請の受付は来年4月から開始される予定で、詳細は公式ウェブサイトで公開されます。
  TextRank (R-L=0.208): 申請の受付は来年4月から開始される予定で、詳細は公式ウェブサイトで公開されます。

[記事 2: ニュース型（冒頭結論）]
  参照要約: 都内の主要鉄道各社は深夜の保線作業時間確保のため、来月1日から終電を約15分繰り上げます。
  リード法     (R-L=0.689): 東京都内の主要鉄道各社は、来月1日から終電時刻を約15分繰り上げるダイヤ改正を実施します。
  TF-IDF   (R-L=0.225): 各社は駅構内のポスターやアプリを通じて、利用者への周知を進めています。
  TextRank (R-L=0.225): 各社は駅構内のポスターやアプリを通じて、利用者への周知を進めています。

[記事 3: ニュース型（冒頭結論）]
  参照要約: A社が充電時間を半分に短縮した新型バッテリーを開発し、年内の量産開始を目指しています。
  リード法     (R-L=0.621): 大手電機メーカーのA社は、従来比で充電時間を半分に短縮した新型バッテリーを開発しました。
  TF-IDF   (R-L=0.621): 大手電機メーカーのA社は、従来比で充電時間を半分に短縮した新型バッテリーを開発しました。
  TextRank (R-L=0.621): 大手電機メーカーのA社は、従来比で充電時間を半分に短縮した新型バッテリーを開発しました。

[記事 4: 後半結論型（課題→調査→結論）]
  参照要約: 住民調査の結果を踏まえ、市は商店街の空き店舗にコワーキングスペースと夜カフェを誘致する施策を決定しました。
  リード法     (R-L=0.304): 私たちの街では近年、駅前商店街の空き店舗が増加していることが大きな悩みでした。
  TF-IDF   (R-L=0.

### 観察とまとめ

1. **ニュース型（冒頭結論）ではリード法が非常に強い**: 新聞・ニュース記事は最初の文に要点をまとめる慣習（逆ピラミッド構造）があるため、単純なリード法が強力なベースラインになります（書籍 4-1-1）。
2. **後半結論型では内容ベースの手法が有利**: 課題や実験の経緯を述べてから最後に結論を書く構成では、リード法は導入文を選んでしまいますが、TF-IDF や TextRank は記事全体のキーワードが集約された結論の文を拾えることがあります。
3. **抽出型要約の限界**: 抽出型要約は文をそのままコピーするため文法的な破綻は起きにくい一方、参照要約のように複数の文の要素を 1 文に圧縮・言い換えすること（**抽象型要約**）はできません。